# TE.ipynb
This notebook sets up the files necessary for running TE! Run this first before running run_TE.sh

In [1]:
import subprocess
import os
import xarray as xa

In [2]:
# The paths for input/output for running TE
tempest_path = '/glade/u/home/ullrich/tempestextremes/bin/'
input_path = '/glade/derecho/scratch/tcorrie/regrids/'
output_path = '/glade/work/tcorrie/ARdata/ARmasks/TE/'
temp_path = input_path+"temp/"

In [3]:
# Add to input files: Only add regridded files for TE not specific for tARget
with open('input_files.txt', 'w') as fin:
    in_list = [input_path+f"{fname}\n" for fname in sorted(os.listdir('/glade/derecho/scratch/tcorrie/regrids/')) if 'ivt' in fname and 'tARget' not in fname]
    fin.writelines(in_list)

# This file will convert the input files in 5D to 3D (see next cell)
with open('input_files_squeezed.txt', 'w') as fin:
    in_list = [temp_path+f"{fname}\n" for fname in sorted(os.listdir('/glade/derecho/scratch/tcorrie/regrids/')) if 'ivt' in fname and 'tARget' not in fname]
    fin.writelines(in_list)

# Generate the outfile list for output TE files
with open('output_files.txt', 'w') as fout:
    out_list = [output_path+f"ARmask.{fname.split('.')[2]}.{fname.split('.')[3]}.nc\n" for fname in in_list]
    fout.writelines(out_list)

In [4]:
# Only run this if you need the 3-D regridded ivt netcdf files. (set run_this = True if necessary)
run_this = False
with open('input_files.txt') as fin, open('input_files_squeezed.txt') as fout:
    files_in = fin.readlines()
    files_out = fout.readlines()
    for file_in, file_out in zip(files_in, files_out):
        if run_this:
            ds_5d = xa.open_dataset(file_in.strip())
            ds_3d = ds_5d.squeeze(drop=True)
            ds_3d.to_netcdf(file_out.strip())

In [5]:
threshold_cmd = '_LAPLACIAN{8,10}(_VECMAG(ivtx,ivty)),<=,-20000,0'

geofilter_cmd = 'area,>=,4e5km2'

command_args = ['/glade/work/tcorrie/conda-envs/TempestEnv/bin/DetectBlobs', 
                '--in_data_list', 'input_files_squeezed.txt', 
                '--out_list', 'output_files.txt',
                '--thresholdcmd', f'\"{threshold_cmd}\"',
                '--minlat', '31.5',
                '--geofiltercmd', f'\"{geofilter_cmd}\"',
                '--lonname', 'lon',
                '--latname', 'lat',
                '--tagvar', 'ARmask',
                '--regional']

#subprocess.run(command_args) # Unless you have a very small dataset, run the generated .sh file below in a linux terminal instead.
# Recommended run: 
# ./run_TE.sh > output_TE.log 2>&1 &
# If you're expecting bad internet: 
# nohup ./run_TE.sh > output_TE.log 2>&1 &

with open('run_TE.sh', 'w') as fsh:
    fsh.write('#!/bin/sh\n')
    fsh.write('# Make sure you have the right path to DetectBlobs pointed and activate an environment with TempestExtremes installed before running this .sh script!')
    fsh.write('\n\n')
    fsh.write(" ".join(command_args))
    #fsh.write('\n\n')
    #fsh.write(f'rm -r {temp_path}') # If you need space, uncomment this line before running TE (it will remove the 3d netcdf files after running)

if not os.access('./run_TE.sh', os.X_OK):
    subprocess.run(['chmod', '+x', 'run_TE.sh'])

In [ ]:
print(" ".join(command_args))

In [ ]:
#result = subprocess.run(['./run_TE.sh'], capture_output=True, text=True)
#print("Output:", result.stdout.strip())
#print(f"Result: {result.stderr}")